# 06. Koncowa prezentacja i raport

**Etap z planu pracy:** koncowa prezentacja.

Notebook tworzy zarys finalnej prezentacji i skondensowany raport wynikow na podstawie tabel zapisanych w poprzednich etapach.


In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "database").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "database" / "NajnowszaWersjaBazy1205_prepared.csv"
RAW_DATA_PATH = PROJECT_ROOT / "database" / "NajnowszaWersjaBazy1205.csv"
METADATA_PATH = PROJECT_ROOT / "outputs" / "prepared_dataset_metadata.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "czysta_baza"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)
print("Prepared data exists:", DATA_PATH.exists())
print("Output dir:", OUTPUT_DIR)

assert DATA_PATH.exists(), f"Brakuje pliku z przygotowana baza: {DATA_PATH}"


PROJECT_ROOT: C:\Users\szymon\projekt_reddit
DATA_PATH: C:\Users\szymon\projekt_reddit\database\NajnowszaWersjaBazy1205_prepared.csv
Prepared data exists: True
Output dir: C:\Users\szymon\projekt_reddit\outputs\czysta_baza


In [2]:
decisions = pd.read_csv(OUTPUT_DIR / "05_hypothesis_decisions.csv")
h3 = pd.read_csv(OUTPUT_DIR / "04_h3_multimodal_benchmark.csv")
best_h3 = h3.sort_values(["negative_f1", "macro_f1"], ascending=False).iloc[0]

slide_outline = pd.DataFrame([
    {"slide": 1, "title": "Temat i pytanie badawcze", "content": "Negatywne odniesienia miedzy subredditami: emocje, jezyk i siec."},
    {"slide": 2, "title": "Dane", "content": "49 918 rekordow, lata 2013-2017, przygotowany zbior bez brakow w kluczowych kolumnach."},
    {"slide": 3, "title": "Hipotezy", "content": "H1 eskalacja emocjonalna, H2 zlozonosc poznawcza, H3 model multimodalny."},
    {"slide": 4, "title": "Metody", "content": "Test Fishera, Mann-Whitney U, Cohen d, TF-IDF, Logistic Regression, Random Forest, Hugging Face baseline."},
    {"slide": 5, "title": "Wynik H1", "content": decisions.loc[decisions["hypothesis"].str.startswith("H1"), "main_evidence"].iloc[0]},
    {"slide": 6, "title": "Wynik H2", "content": decisions.loc[decisions["hypothesis"].str.startswith("H2"), "main_evidence"].iloc[0]},
    {"slide": 7, "title": "Wynik H3", "content": f"Najlepszy model: {best_h3['model']}; negative_f1={best_h3['negative_f1']:.3f}."},
    {"slide": 8, "title": "Dlaczego accuracy nie wystarcza", "content": "Klasa negatywna jest mniejszosciowa, wiec model moze miec wysoka accuracy i slaby recall/F1 dla klasy 1."},
    {"slide": 9, "title": "Ograniczenia", "content": "Male grupy historii 24h, roznica LINK_SENTIMENT vs Content_Sentiment, koszt modeli transformerowych."},
    {"slide": 10, "title": "Wniosek koncowy", "content": "Najsilniejsze sa sygnaly tekstowe TF-IDF; hipotezy wymagaja ostroznej interpretacji i dalszych rozszerzen."},
])
display(slide_outline)
slide_outline.to_csv(OUTPUT_DIR / "06_slide_outline.csv", index=False)


,slide,title,content
0,1,Temat i pytanie badawcze,Negatywne odniesienia miedzy subredditami: emo...
1,2,Dane,"49 918 rekordow, lata 2013-2017, przygotowany ..."
2,3,Hipotezy,"H1 eskalacja emocjonalna, H2 zlozonosc poznawc..."
3,4,Metody,"Test Fishera, Mann-Whitney U, Cohen d, TF-IDF,..."
4,5,Wynik H1,"negative_rate high anger=0.0726, other=0.0772,..."
5,6,Wynik H2,2/6 cech ma nizsza wartosc dla linkow negatywnych
6,7,Wynik H3,Najlepszy model: TF-IDF word 1-2 + Logistic Re...
7,8,Dlaczego accuracy nie wystarcza,"Klasa negatywna jest mniejszosciowa, wiec mode..."
8,9,Ograniczenia,"Male grupy historii 24h, roznica LINK_SENTIMEN..."
9,10,Wniosek koncowy,Najsilniejsze sa sygnaly tekstowe TF-IDF; hipo...


In [3]:
report = [
    "# Raport koncowy - badanie hipotez na oczyszczonej bazie",
    "",
    "## Dane",
    "",
    "- Plik: `database/NajnowszaWersjaBazy1205_prepared.csv`.",
    "- Liczba obserwacji: 49 918.",
    "- Target: `is_negative_link`, czyli binarna wersja `LINK_SENTIMENT`.",
    "- Podzial: `split_chronological`, aby test odzwierciedlal predykcje na pozniejszych danych.",
    "",
    "## Decyzje wobec hipotez",
    "",
]

for row in decisions.itertuples(index=False):
    report.append(f"- **{row.hypothesis}: {row.decision}.** {row.main_evidence}.")

report.extend([
    "",
    "## Najlepszy model H3",
    "",
    f"- Model: `{best_h3['model']}`.",
    f"- Accuracy: {best_h3['accuracy']:.3f}.",
    f"- Macro F1: {best_h3['macro_f1']:.3f}.",
    f"- F1 klasy negatywnej: {best_h3['negative_f1']:.3f}.",
    "",
    "## Materialy",
    "",
    "- Notebooki: `notebooks_czysta_baza/`.",
    "- Tabele i wykresy: `outputs/czysta_baza/`.",
])

report_text = "\n".join(report) + "\n"
(OUTPUT_DIR / "06_raport_koncowy.md").write_text(report_text, encoding="utf-8")
display(Markdown(report_text))


# Raport koncowy - badanie hipotez na oczyszczonej bazie

## Dane

- Plik: `database/NajnowszaWersjaBazy1205_prepared.csv`.
- Liczba obserwacji: 49 918.
- Target: `is_negative_link`, czyli binarna wersja `LINK_SENTIMENT`.
- Podzial: `split_chronological`, aby test odzwierciedlal predykcje na pozniejszych danych.

## Decyzje wobec hipotez

- **H1 eskalacja emocjonalna: niepotwierdzona.** negative_rate high anger=0.0726, other=0.0772, Fisher p=1.0000, odds ratio=0.935.
- **H2 zlozonosc poznawcza: czesciowo potwierdzona.** 2/6 cech ma nizsza wartosc dla linkow negatywnych.
- **H3 model multimodalny: niepotwierdzona.** najlepszy model: TF-IDF word 1-2 + Logistic Regression balanced, negative_f1=0.3437.

## Najlepszy model H3

- Model: `TF-IDF word 1-2 + Logistic Regression balanced`.
- Accuracy: 0.856.
- Macro F1: 0.631.
- F1 klasy negatywnej: 0.344.

## Materialy

- Notebooki: `notebooks_czysta_baza/`.
- Tabele i wykresy: `outputs/czysta_baza/`.


## Rezultat etapu

Powstal szkic narracji do prezentacji oraz krotki raport markdown. Pelne dowody liczbowe sa w notebookach 01-05 i w tabelach `outputs/czysta_baza`.
